# Brand list curation

Brand-list build, website checks, category labels, and files for later steps.


In [1]:
from pathlib import Path
from datetime import date, datetime
import re, json, time, random, html as html_lib, unicodedata
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
try:
    import requests
    from bs4 import BeautifulSoup
except ImportError as exc:
    raise ImportError('requests and beautifulsoup4 are not available') from exc

DATA_DIR   = Path('.')
OUTPUT_CSV = DATA_DIR / 'brands_catalog.csv'
TODAY      = date.today().isoformat()
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 120)

# Use curl_cffi for sites that block normal requests.
USE_CURL_CFFI = False
try:
    from curl_cffi import requests as cffi_requests
    USE_CURL_CFFI = True
except Exception:
    cffi_requests = None
print('HTTP backend:', 'curl_cffi (browser-TLS)' if USE_CURL_CFFI
      else 'requests')

HTTP backend: curl_cffi (browser-TLS)


## Sources

Brand-name sources, plus a small hand-added seed list.


In [ ]:
SOURCE_REGISTRY = pd.DataFrame([
    ('broad_multi_brand_retailer', 'Farfetch designers',    1, 'https://www.farfetch.com/designers/women'),
    ('broad_multi_brand_retailer', 'END. brands',           1, 'https://www.endclothing.com/row/brands'),
    ('broad_multi_brand_retailer', 'LN-CC brands',          1, 'https://www.ln-cc.com/en-pl/brands-all'),
    ('broad_multi_brand_retailer', 'Browns designers',      1, 'https://brownsfashion.com/pages/designers'),
    ('broad_multi_brand_retailer', 'Selfridges directory',  2, 'https://www.selfridges.com/GB/en/brand-directory/'),
    ('broad_multi_brand_retailer', 'Net-a-Porter A-Z',      2, 'https://www.net-a-porter.com/en-nl/shop/azdesigners'),
    ('broad_multi_brand_retailer', 'MR PORTER designers',   2, 'https://www.mrporter.com/en-us/mens/azdesigners'),
    ('broad_multi_brand_retailer', '24S designers',         2, 'https://www.24s.com/en-nl/women/brands'),
    ('niche_concept_store', 'Dover Street Market London',   1, 'https://shop.doverstreetmarket.com/collections'),
    ('niche_concept_store', 'Dover Street Market NY',       1, 'https://shop-us.doverstreetmarket.com/collections'),
    ('niche_concept_store', 'MACHINE-A',                    1, 'https://www.machine-a.com/'),
    ('niche_concept_store', 'APOC Store designers',         1, 'https://apoc-store.com/pages/designers'),
    ('niche_concept_store', 'Garmentory designers',         2, 'https://www.garmentory.com/designers'),
    ('niche_concept_store', 'Wolf & Badger new',            2, 'https://www.wolfandbadger.com/global/designers/new/'),
    ('niche_concept_store', 'Slam Jam brands',              2, 'https://slamjam.com/en-de/pages/brands'),
    ('niche_concept_store', 'Goodhood mens',                2, 'https://goodhoodstore.com/en-nl/pages/mens-brands'),
    ('niche_concept_store', 'Goodhood womens',              2, 'https://goodhoodstore.com/pages/womens-brands'),
    ('niche_concept_store', 'The Broken Arm brands',        2, 'https://www.the-broken-arm.com/en/brands'),
    ('contemporary_sustainable', 'Verishop brands',         2, 'https://www.verishop.com/brands'),
    ('contemporary_sustainable', 'Revolve all brands',      2, 'https://www.revolve.com/designers/?navsrc=designers'),
    ('contemporary_sustainable', 'Shop Like You Give a Damn brands', 2, 'https://www.shoplikeyougiveadamn.com/brands'),
    ('contemporary_sustainable', 'ASOS A-Z women',          3, 'https://www.asos.com/women/a-to-z-of-brands/cat/?cid=1340'),
    ('contemporary_sustainable', 'ASOS A-Z men',            3, 'https://www.asos.com/men/a-to-z-of-brands/cat/?cid=1361'),
    ('incubator_award', 'Fashion East',                     1, 'https://www.fashioneast.co.uk/designers/'),
    ('incubator_award', 'BFC NEWGEN',                       1, 'https://www.britishfashioncouncil.co.uk/BFC-Initiatives/BFC-NEWGEN'),
    ('incubator_award', 'LVMH Prize',                       1, 'https://www.lvmhprize.com/en'),
    ('incubator_award', 'Woolmark Prize',                   1, 'https://www.woolmarkprize.com/designers/'),
    ('incubator_award', 'ANDAM',                            1, 'https://www.andam.fr/'),
    ('incubator_award', 'CFDA/Vogue Fashion Fund',          2, 'https://cfda.com/programs/designers/cfda-vogue-fashion-fund'),
    ('editorial_runway', 'Vogue Runway designers',          1, 'https://www.vogue.com/fashion-shows/designers'),
    ('editorial_runway', '1 Granary',                       2, 'https://1granary.com/'),
], columns=['source_family', 'source_name', 'priority', 'url'])

# Use sitemaps for JS-heavy sites.
SITEMAP_SOURCES = ('net-a-porter.com', 'mrporter.com')

# Keep these as evidence only, not scraped names.
EVIDENCE_ONLY = {'Vogue Runway designers', '1 Granary'}

# Extra brands added by hand. Duplicates are removed later.
SEED_BRANDS = [
    # UK / Europe
    'Lucy & Yak', "Nobody's Child", 'Sisterhood', 'House of Sunny', 'Damson Madder',
    'Kitri', 'Rixo', 'Omnes', 'Olivia Rubin', 'Never Fully Dressed', 'Sezane', 'Rouje',
    'Realisation Par', 'With Jean', 'Faithfull the Brand', 'Baukjen', 'Albaray', 'Cefinn',
    'ME+EM', 'Aligne', 'Lisou', 'Stella Nova', 'Whistles', 'Ghost', 'Thought Clothing',
    'People Tree', 'Organic Basics', 'Colorful Standard', 'Asket', 'Finisterre',
    'Paloma Wool', 'Nu-In', 'Mud Jeans', 'Armedangels', 'Glassworks London',
    # North America
    'Reformation', 'Free People', 'Everlane', 'Quince', 'Cuyana', 'Christy Dawn',
    'Mate the Label', 'Outerknown', 'Girlfriend Collective', 'Lisa Says Gah',
    'Kotn', 'Pangaia', 'Tentree', 'Doen', 'Vuori', 'Tradlands',
    # Footwear
    'VIVAIA', 'Allbirds', "Rothy's", 'Cariuma', 'Nisolo', 'Atoms', 'Vivobarefoot',
    'Koio', 'Margaux', 'Sarah Flint', 'Birdies', 'Vagabond', 'Hereu', 'Cano',
    'Thousand Fell',
    # Bags
    'Mlouye', 'Polène', 'Strathberry', 'DeMellier', 'Senreve', 'Dagne Dover',
    'JW PEI', 'Cafune', 'Telfar', 'Little Liffner', 'Marge Sherwood', 'Marhen.J',
    'Freitag', 'Baggu', 'Bellroy', 'Dragon Diffusion', 'Wandler', 'Yuzefi',
    # Jewellery
    'Mejuri', 'Astrid & Miyu', 'Missoma', 'Monica Vinader', 'Ana Luisa', 'PDPAOLA',
    'Otiumberg', 'Daphine', 'Aurate',
    # China / Hong Kong / Singapore
    'VIVAIA', 'Neiwai', 'Maia Active', 'Particle Fever', 'Songmont', 'Short Sentence',
    'Roaringwild', 'Beaster', 'Matter Prints',
    # Korea
    'Matin Kim', 'Mardi Mercredi', 'Recto', 'Nonlocal', 'Kirsh', 'Depound',
    # Japan
    'CLANE', 'Foufou', 'Snow Peak',
    # South / Southeast Asia
    'Nicobar', 'Doodlage', 'No Nasties', 'Suta', 'Okhai', 'The Summer House',
    'Bhaane', 'Sui',
    # Middle East
    'Okhtein', 'Bouguessa', "L'Afshar", 'Nafsika Skourti', 'Bil Arabi',
    # Africa
    'Maxhosa Africa', 'Rich Mnisi', 'Tongoro', 'Orange Culture', 'Andrea Iyamah',
    'Christie Brown', 'Lisa Folawiyo', 'Kente Gentlemen',
    # Latin America
    'Maygel Coronel', 'Farm Rio', 'Lenny Niemeyer', 'Agua Bendita', 'Haight',
    # Australia / New Zealand
    'Spell', 'Arnhem', 'Kowtow', 'Nagnata', 'Dissh', 'Bondi Born',
    'Viktoria & Woods', 'Anna Quan',
]
print(f'{len(SOURCE_REGISTRY)} sources ({len(EVIDENCE_ONLY)} evidence-only), '
      f'{len(set(SEED_BRANDS))} unique seed brands')
SOURCE_REGISTRY

## Helpers

Helper functions for scraping, cleaning, and category labels.


In [ ]:
from pathlib import Path

helper_path = Path("helpers/curation_helpers.py")
if not helper_path.exists():
    helper_path = Path("step_1_curate_dataset/helpers/curation_helpers.py")
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())


## Build list

Source names, seed names, and duplicate cleanup.


In [ ]:
RUN_LIVE_QUERY = True   # False => skip network, seeds only
SOURCE_LIMIT   = None   # e.g. 3 for a fast smoke test

if RUN_LIVE_QUERY:
    live, query_log = query_sources(SOURCE_REGISTRY, limit=SOURCE_LIMIT)
else:
    live = pd.DataFrame(columns=['brand_name', 'canonical_key', 'source_name', 'source_family'])
    query_log = pd.DataFrame()
print(f'\n{len(live):,} raw candidate rows from {live["source_name"].nunique() if len(live) else 0} sources')

# One row per brand.
live = live[live['canonical_key'].str.len() > 0].copy()
live['_len'] = live['brand_name'].str.len()
scraped = (live.sort_values(['canonical_key', '_len', 'brand_name'])
               .drop_duplicates('canonical_key', keep='first')[['brand_name', 'canonical_key']]
               .assign(trusted=False))

# Add seed brands not already found.
seed_all = pd.DataFrame({'brand_name': sorted(set(SEED_BRANDS))})
seed_all['canonical_key'] = seed_all['brand_name'].map(canonical_key)
is_covered = seed_all['canonical_key'].isin(scraped['canonical_key'])
seed_df = seed_all[~is_covered].assign(trusted=True)

# Show which seed brands were new.
covered = sorted(seed_all[is_covered]['brand_name'], key=str.lower)
netnew  = sorted(seed_df['brand_name'], key=str.lower)
print(f'\nseeds: {len(netnew)} net-new (true gaps), {len(covered)} already found by scraping')
if covered:
    print('  already covered (seed redundant): ' + ', '.join(covered))
if netnew:
    print('  net-new from seeds:              ' + ', '.join(netnew))

brands = (pd.concat([scraped, seed_df], ignore_index=True)
            .drop_duplicates('canonical_key', keep='first')
            .sort_values('brand_name', key=lambda s: s.str.lower())
            .reset_index(drop=True))
print(f'\n{len(brands):,} unique brands ({int(brands.trusted.sum())} from seed list)')
brands.head(20)

## Websites

Website matches and domain checks.


In [ ]:
RESOLVE_WEBSITES = True
seed_site_by_key = {canonical_key(k): v for k, v in SEED_WEBSITES.items()}
if RESOLVE_WEBSITES:
    print(f'Resolving {len(brands):,} websites '
          f'({len(seed_site_by_key)} via manual override, rest via Clearbit)...', flush=True)
    sites = []
    for i, nm in enumerate(brands['brand_name'], 1):
        override = seed_site_by_key.get(canonical_key(nm))
        if override:
            sites.append(override)          # known URL -> no network call
        else:
            sites.append(resolve_website(nm))
            time.sleep(0.15)                # only throttle real Clearbit calls
        if i % 50 == 0 or i == len(brands):
            print(f'  [{i}/{len(brands)}] found {sum(bool(s) for s in sites)}', flush=True)
    brands['official_website'] = sites
else:
    brands['official_website'] = ''

before = len(brands)
brands = brands[brands['official_website'] != ''].copy()
# Check scraped website domains.
ok_domain = brands.apply(lambda r: r['trusted'] or name_in_domain(r['brand_name'], r['official_website']), axis=1)
brands = brands[ok_domain].reset_index(drop=True)
print(f'{len(brands):,} brands with a verified website (-{before - len(brands):,})')
brands.head(20)

## Categories

Product-type labels for each brand.


In [ ]:
print(f'Classifying {len(brands):,} sites with {CAT_WORKERS} workers...', flush=True)
cats, done = {}, 0
with ThreadPoolExecutor(max_workers=CAT_WORKERS) as ex:
    futs = {ex.submit(classify_site, u): u for u in brands['official_website'].unique()}
    for fut in as_completed(futs):
        cats[futs[fut]] = fut.result(); done += 1
        if done % 100 == 0 or done == len(futs):
            print(f'  [{done}/{len(futs)}]', flush=True)
brands['category'] = brands['official_website'].map(cats).fillna('unknown')

# Remove unknown scraped rows.
before = len(brands)
brands = brands[(brands['category'] != 'unknown') | brands['trusted']].reset_index(drop=True)
print(f'kept {len(brands):,} (-{before - len(brands):,} non-fashion / unreachable)')
print(brands['category'].value_counts().to_string())

## Raw file

`brands_catalog.csv`.


In [ ]:
catalog = (brands[['brand_name', 'official_website', 'category']]
           .drop_duplicates('brand_name')
           .sort_values('brand_name', key=lambda s: s.str.lower())
           .reset_index(drop=True))
catalog.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(catalog):,} brands -> {OUTPUT_CSV}')
# Check key brands are present.
present = catalog[catalog['brand_name'].str.contains(r'lucy|yak|nobody|sisterhood', case=False, na=False)]
print('\ncoverage check (previously missing):')
print(present.to_string(index=False) if len(present) else '  (none -- run live query + seeds)')
catalog.head(40)

## Final cleanup

Clean catalogue.


In [ ]:
from pathlib import Path

helper_path = Path("helpers/final_cleanup.py")
if not helper_path.exists():
    helper_path = Path("step_1_curate_dataset/helpers/final_cleanup.py")
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())


## Category files

Separate files for clothes, shoes, bags, and jewellery.


In [ ]:
# Save category files.
import pandas as pd
from pathlib import Path
try:
    DATA_DIR
except NameError:
    DATA_DIR = Path('.')

cat = pd.read_csv(DATA_DIR / 'brands_catalog_clean.csv')

# 'all' goes into every category file.
UNIVERSES = {
    'brands_clothes.csv':   {'clothes', 'all'},
    'brands_shoes.csv':     {'only shoes', 'all'},
    'brands_bags.csv':      {'only bags', 'all'},
    'brands_jewellery.csv': {'only jewellery', 'all'},
}

for fname, cats in UNIVERSES.items():
    sub = (cat[cat['category'].isin(cats)][['brand_name', 'official_website']]
           .drop_duplicates('brand_name')
           .sort_values('brand_name', key=lambda s: s.str.lower())
           .reset_index(drop=True))
    sub.to_csv(DATA_DIR / fname, index=False)
    print(f'{fname:24s} {len(sub):>5} brands', flush=True)
